# Trade the 8-K: executive exits (pooled CEO/CFO/officer departures)

Runs the Massive starter notebook's own pipeline (sections 1-7) on **any window**, with the pooled exec-exit event table.
Requirements: the starter's `.venv` (`gqh-massive-8k-starter/setup.sh`) and `MASSIVE_API_KEY` in `gqh-massive-8k-starter/.env`.

**Judges, for the sealed window:** set `START` / `END` below (or the env vars `START`, `END`) and run all cells.
`EVENTS` selects the event table: `exec` (pooled CEO/CFO/officer departures, our main study), `cfo_appointment`, `ceo_appointment`, `cfo_departure`, `settlement`, or `peers`.
The starter's API cell and all pipeline cells run **unchanged**; only the window and the event table are configured.
Pre-registered hypotheses: `HYPOTHESIS.md` (trial 1, cash-secured put) and `HYPOTHESIS_2.md` (trial 2, post-hoc protective put).
Our sealed-window predictions are hash-stamped in `FORWARD_LEDGER.md` (P1-P4) and `FORWARD_LEDGER_2.md` (P5-P7).

In [ ]:
import os, subprocess, sys
from pathlib import Path
import pandas as pd

START = os.environ.get("START", "2026-01-01")   # judges: change to the sealed window
END = os.environ.get("END", "2026-08-31")
LABEL = os.environ.get("LABEL", "nb")
EVENTS = os.environ.get("EVENTS", "exec")       # exec | cfo_appointment | ceo_appointment | cfo_departure | settlement | peers

HERE = Path.cwd() if (Path.cwd() / "run_window.py").exists() else Path.cwd() / "strategy_8k"
STARTER = HERE.parent / "gqh-massive-8k-starter"
assert (STARTER / ".env").exists(), "put MASSIVE_API_KEY in gqh-massive-8k-starter/.env"
print(START, END, LABEL, STARTER)

In [ ]:
# Runs sections 1-7 of the starter notebook with the pooled exec-exit events. API responses are cached on disk.
proc = subprocess.run([sys.executable, str(HERE / "run_window.py"), "--events", EVENTS, "--start", START, "--end", END,
                       "--label", LABEL], cwd=STARTER, capture_output=True, text=True)
print("\n".join(l for l in proc.stdout.splitlines() if "events" in l or "->" in l))
if proc.returncode:
    print(proc.stderr[-3000:]); raise RuntimeError("pipeline failed")

In [ ]:
out = HERE / "output"
diff = pd.read_pickle(out / f"{LABEL}_diff.pkl")
pd.set_option("display.width", 200)
for s in ["cash_secured_put", "protective_put", "long_call", "stock"]:
    d = diff[diff.strategy == s].set_index("horizon")[["n_a", "n_b", "difference", "ci_lo", "ci_hi"]]
    print(f"\n{s}: events minus placebo, P&L per $1 of spot (3-6m options, entry=post, OTM 5%)")
    print((d.assign(**{c: d[c].map(lambda v: f"{v:+.2%}") for c in ["difference", "ci_lo", "ci_hi"]})).to_string())

In [ ]:
# Prediction check: |realized| / implied move, events vs ordinary days (3-6m ATM straddle read on the pre-event session)
import numpy as np
def ratio(res):
    r = res[(res.bucket == "3-6m") & (res.entry == "pre") & (res.otm == 0.05)]
    return r.groupby("horizon")["ratio"].agg(["count", "mean", "median"])
ev, pl = pd.read_pickle(out / f"{LABEL}_results.pkl"), pd.read_pickle(out / f"{LABEL}_placebo_results.pkl")
print(pd.concat({"events": ratio(ev), "placebo": ratio(pl)}, axis=1).round(2).to_string())